# 🗓️ Project 13.1 — Robot Lab Interval Scheduler
**DSA for Mechatronics · Week 13 — Greedy Algorithms**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, heapq
from collections import defaultdict
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A shared robot lab has limited testing slots and charging stations.
We solve three classic interval problems with greedy algorithms:

1. **Activity selection** — maximise the number of non-overlapping tests
   (sort by finish time — the fundamental greedy interval algorithm)
2. **Interval partitioning** — find the minimum number of charging stations
   needed so all robots can charge without conflict
3. **Minimum arrows** — fewest "shutdown pulses" to halt all active robots
   whose activity windows overlap the pulse time


## Step 1 — Generate interval dataset

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
N_ACTS   = random.randint(10, 16)
N_CHARGE = random.randint(10, 16)
N_ROBOTS = random.randint(10, 16)
T_MAX    = random.randint(20, 30)

def rand_interval(t_max):
    s = random.randint(0, t_max - 2)
    e = random.randint(s + 1, t_max)
    return (s, e)

ACTS    = [rand_interval(T_MAX) for _ in range(N_ACTS)]
CHARGE  = [rand_interval(T_MAX) for _ in range(N_CHARGE)]
ROBOTS  = [rand_interval(T_MAX) for _ in range(N_ROBOTS)]

print(f"Interval datasets:")
print(f"  T_MAX = {T_MAX}")
print()
print(f"  Activities (n={N_ACTS}) — for activity selection:")
for i, (s, e) in enumerate(ACTS):
    bar = " " * s + "█" * (e - s)
    print(f"    [{i:2d}] ({s:2d},{e:2d})  |{bar:<{T_MAX}}|")
print()
print(f"  Charge slots (n={N_CHARGE}) — for interval partitioning:")
for i, (s, e) in enumerate(CHARGE):
    bar = " " * s + "█" * (e - s)
    print(f"    [{i:2d}] ({s:2d},{e:2d})  |{bar:<{T_MAX}}|")
print()
print(f"  Robot windows (n={N_ROBOTS}) — for minimum arrows:")
for i, (s, e) in enumerate(ROBOTS):
    bar = " " * s + "█" * (e - s)
    print(f"    [{i:2d}] ({s:2d},{e:2d})  |{bar:<{T_MAX}}|")


## Step 2 — Activity selection (maximise non-overlapping tasks)

In [ ]:
def activity_selection(intervals):
    """
    Select the maximum number of non-overlapping activities.

    Greedy strategy: ALWAYS pick the activity that finishes earliest.
    Proof (exchange argument): suppose OPT picks activity a instead of our
    greedy choice g (which ends first). Swapping a for g in OPT cannot
    make it worse — g ends no later than a, so any activity compatible
    with a is also compatible with g.

    Algorithm:
      1. Sort by finish time.
      2. Always select the next activity whose start ≥ last finish.
    O(n log n) for sort, O(n) for selection.
    """
    sorted_acts = sorted(enumerate(intervals), key=lambda x: x[1][1])
    selected = []
    last_end  = -1
    for orig_idx, (s, e) in sorted_acts:
        if s >= last_end:
            selected.append((orig_idx, s, e))
            last_end = e
    return selected, sorted_acts

selected, sorted_acts = activity_selection(ACTS)

print(f"Activity selection (n={N_ACTS}):")
print(f"  Sorted by finish time:")
for orig_i, (s, e) in sorted_acts:
    chosen = "✅ SELECTED" if any(i == orig_i for i, _, _ in selected) else "  skipped "
    print(f"    [{orig_i:2d}] ({s:2d},{e:2d})  {chosen}")
print()
print(f"  Selected activities   : {[(i, s, e) for i, s, e in selected]}")
print(f"  Count selected        : {len(selected)} out of {N_ACTS}")
print()
# Verify non-overlapping
ok = all(selected[i+1][1] >= selected[i][2] for i in range(len(selected)-1))
print(f"  Non-overlapping check : {'✅ YES' if ok else '❌ NO'}")


## Step 3 — Interval partitioning (minimum parallel resources)

In [ ]:
def interval_partitioning(intervals):
    """
    Minimum number of 'rooms' (parallel resources) so all intervals can run.

    Greedy strategy: process intervals in order of START time.
    Maintain a min-heap of current end times (one per room).
      - If earliest-ending room finishes before new interval starts → reuse it.
      - Otherwise → open a new room.

    Key insight: the minimum rooms needed equals the maximum number of
    simultaneously overlapping intervals (depth of coverage).

    O(n log n) for sort + heap operations.
    """
    sorted_ch = sorted(enumerate(intervals), key=lambda x: x[1][0])
    heap = []           # min-heap of (end_time, room_id)
    room_assign = {}    # orig_idx → room_id
    n_rooms = 0
    for orig_idx, (s, e) in sorted_ch:
        if heap and heap[0][0] <= s:
            end_t, room_id = heapq.heappop(heap)
            heapq.heappush(heap, (e, room_id))
            room_assign[orig_idx] = room_id
        else:
            n_rooms += 1
            heapq.heappush(heap, (e, n_rooms))
            room_assign[orig_idx] = n_rooms
    return n_rooms, room_assign

min_rooms, room_assign = interval_partitioning(CHARGE)

# Max depth verification
def max_overlap_depth(intervals):
    events = []
    for s, e in intervals:
        events.append((s, +1))
        events.append((e, -1))
    events.sort()
    depth = cur = 0
    for _, d in events:
        cur += d
        depth = max(depth, cur)
    return depth

depth = max_overlap_depth(CHARGE)

print(f"Interval partitioning — minimum charging stations (n={N_CHARGE}):")
print()
print(f"  Assignments (interval → station):")
n_rooms_found = max(room_assign.values()) if room_assign else 0
for room in range(1, n_rooms_found + 1):
    assigned = [(i, CHARGE[i][0], CHARGE[i][1])
                for i, r in room_assign.items() if r == room]
    assigned.sort(key=lambda x: x[1])
    print(f"    Station {room}: {[(f'[{i}]({s},{e})') for i, s, e in assigned]}")
print()
print(f"  Minimum stations needed : {min_rooms}")
print(f"  Max simultaneous depth  : {depth}")
print(f"  Correct (min==depth)    : {'✅ YES' if min_rooms == depth else '❌ NO'}")


## Step 4 — Minimum arrows to burst all robot windows

In [ ]:
def min_arrows(intervals):
    """
    Minimum number of vertical arrows to pierce all intervals.
    Equivalent to: minimum number of points that hit every interval.

    Greedy strategy: sort by END point.
      - Shoot the first arrow at the end of the earliest-ending interval.
      - This arrow hits all intervals containing that end point.
      - Remove all hit intervals; repeat for the remaining ones.

    Exchange argument: shooting at the END of the current earliest interval
    is never worse than shooting anywhere else within it — shooting later
    would miss some intervals that start after this end.

    O(n log n) for sort, O(n) scan.
    """
    if not intervals: return 0, []
    sorted_r = sorted(intervals, key=lambda x: x[1])
    arrows = []
    arrow_pos = sorted_r[0][1]
    arrows.append(arrow_pos)
    for s, e in sorted_r[1:]:
        if s > arrow_pos:               # this interval not hit → new arrow
            arrow_pos = e
            arrows.append(arrow_pos)
    # Verify: each interval contains at least one arrow
    hits = []
    for arrow in arrows:
        burst = [(s, e) for s, e in intervals if s <= arrow <= e]
        hits.append((arrow, len(burst)))
    return len(arrows), arrows, hits

n_arrows, arrow_positions, hit_detail = min_arrows(ROBOTS)

print(f"Minimum arrows — pierce all robot windows (n={N_ROBOTS}):")
print()
print(f"  Robot windows sorted by end:")
sorted_robs = sorted(enumerate(ROBOTS), key=lambda x: x[1][1])
for orig_i, (s, e) in sorted_robs:
    hit = any(s <= a <= e for a in arrow_positions)
    print(f"    [{orig_i:2d}] ({s:2d},{e:2d})  {'✅ hit' if hit else '❌ MISS'}")
print()
print(f"  Arrow positions  : {arrow_positions}")
print(f"  Arrows needed    : {n_arrows}")
print()
print(f"  Per-arrow coverage:")
for pos, count in hit_detail:
    print(f"    Arrow at t={pos:3d} → hits {count} intervals")
all_hit = all(any(s <= a <= e for a in arrow_positions) for s, e in ROBOTS)
print()
print(f"  All intervals hit: {'✅ YES' if all_hit else '❌ NO'}")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " ROBOT LAB INTERVAL SCHEDULER — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  ACTIVITY SELECTION" + " "*(W-20) + "║")
print(f"║  {'Activities (n)':<28}: {N_ACTS:<26}║")
print(f"║  {'Selected':<28}: {len(selected)} of {N_ACTS}{'':<20}║")
sel_str = str([(i, s, e) for i, s, e in selected[:3]]) + ("..." if len(selected)>3 else "")
print(f"║  {'Non-overlapping':<28}: {'YES ✅' if ok else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  INTERVAL PARTITIONING" + " "*(W-23) + "║")
print(f"║  {'Intervals (n)':<28}: {N_CHARGE:<26}║")
print(f"║  {'Min stations':<28}: {min_rooms:<26}║")
print(f"║  {'Max depth verify':<28}: {depth:<26}║")
print(f"║  {'Correct':<28}: {'YES ✅' if min_rooms == depth else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  MINIMUM ARROWS" + " "*(W-16) + "║")
print(f"║  {'Robot windows (n)':<28}: {N_ROBOTS:<26}║")
print(f"║  {'Arrows needed':<28}: {n_arrows:<26}║")
print(f"║  {'Arrow positions':<28}: {str(arrow_positions):<26}║")
print(f"║  {'All intervals hit':<28}: {'YES ✅' if all_hit else 'NO ❌':<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about greedy algorithms in this context?

*Your answer here:*

---

**Q2 — Why does the greedy choice work here?**
For one of the algorithms in this project, explain the *exchange argument* — why swapping the greedy choice for any other choice cannot improve the solution.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
